# 12 - Memory Within a Conversation (Checkpointers & Threads)

## Purpose

This notebook demonstrates a LangChain DeepAgents tutorial episode. It walks through the key concepts and code patterns for building, configuring, and operating AI agents with the `deepagents` framework.

## What You'll Learn

- How deep agents differ from regular LangChain agents
- The built-in tool suite every agent receives automatically
- How to configure models, backends, permissions, and memory
- Practical patterns for tools, planning, memory, and handoffs

## Tutorial Series Context

This is Episode 12-checkpointer_threads of the LangChain DeepAgents YouTube tutorial series. Each episode focuses on a specific aspect of agent development, from basic introduction to advanced patterns like subagents, memory, permissions, and skills.

## How to Use This Notebook

1. **Run cells top-to-bottom**: The code is designed to execute in order, with each section building on the previous one.
2. **Observe tool calls**: Watch for `tool_calls` in message outputs - this shows which agents/tools are being used.
3. **Modify and experiment**: The code is intentionally kept identical to the original Python scripts. Feel free to modify parameters, models, or prompts to see how behavior changes.
4. **Refer to comments**: Each code section has accompanying markdown explanations of what the code does and why.

## Key Concepts Covered in This Episode

- InMemorySaver keeps history in RAM for demos and tests
- SqliteSaver or PostgresSaver for persistence across restarts
- checkpointer saves state after every step
- thread_id names the conversation; same id = same history
- Different thread_id = separate conversation (isolation)
- get_state() reads back whatever checkpointer holds for a thread


# Overview: 12-checkpointer_threads.ipynb

This notebook converts the Python tutorial `12-checkpointer_threads.ipynb` into an interactive Jupyter Notebook format, preserving all code examples and explanations exactly as they appear in the original tutorial scripts.

## Notebook Structure

Each logical section of the original Python script has been split into its own code cell. Each code cell is preceded by a markdown cell containing:

- **A short heading** describing what that section does
- **1–3 sentences** explaining the purpose and key concepts of that section

## How to Navigate

- **Run cells in order**: The tutorial was designed to be executed top-to-bottom. Each section builds on the previous one.
- **Observe tool calls**: Watch the output for `tool_calls` entries - this shows which built-in or tools the agent invokes.
- **Read the markdown comments**: Each code section has a preceding explanation of what the code does and why it's structured that way.
- **Experiment freely**: The original `.py` files are unchanged. Feel free to modify, extend, or experiment with the code in this notebook.

## Educational Value

This notebook serves as both a working tutorial replication and a learning resource. By running each cell and reading the accompanying explanations, you'll gain hands-on experience with:

- LangChain DeepAgents framework architecture
- Backend configuration (virtual vs real filesystem)
- Tool definition and registration
- Planning, memory, permissions, and context management
- Subagent delegation and structured output
- Streaming modes and human-in-the-loop patterns

## Tips for Getting the Most Out of This Notebook

1. **Don't skip the markdown cells**: The educational content between code sections is essential for understanding the "why" behind the code.
2. **Re-run after modifications**: After experimenting with changes, re-run all cells from the top to ensure everything still works together.
3. **Refer to the original Python scripts**: The `.py` files in the same directory are unchanged and can be referenced for the exact original code.
4. **Use rich output**: This notebook uses Rich formatting for colorful console output - pay attention to color-coded hints and status messages.

---

_Notebook automatically generated from `12-checkpointer_threads.ipynb`. Original Python scripts preserved unchanged._


## Section 2: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section sets up the imports and environment configurati
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
"""
12 - Memory Within a Conversation (Checkpointers & Threads)
===========================================================
- A checkpointer saves state after every step, so turn 2 remembers turn 1
- A `thread_id` names the conversation; same id = same history
- Different thread_id = a completely separate conversation
Run:  python 12-checkpointer_threads.py
"""

## Section 4: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 1: Imports and setup
import os

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver  # dev checkpointer
from rich import print

load_dotenv()
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")


def text_of(message) -> str:
    """Normalize message content across providers (str vs content blocks)."""
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(block.get("text", "") for block in content if isinstance(block, dict))

## Section 6: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section explains checkpointer configuration and thread-
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 2: Add a checkpointer
# InMemorySaver keeps history in RAM — great for demos and tests. Swap in
# SqliteSaver or PostgresSaver for something that survives a restart.
agent = create_deep_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
    system_prompt="You are a friendly assistant. Reply in one short sentence.",
)


def say(text: str, thread_id: str) -> str:
    """Send one message on a named thread and return the reply."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config=config)
    return text_of(result["messages"][-1]).strip()

## Section 8: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 3: Two turns on the SAME thread
# The second invoke passes no history — the checkpointer reloads it from
# thread "alice". This is what makes a real chat loop possible.
print("[bold cyan]Thread 'alice':[/bold cyan]")
print("  [dim]user:[/dim]  My favorite number is 42.")
print(f"  [green]agent:[/green] {say('My favorite number is 42.', 'alice')}")
print("  [dim]user:[/dim]  What is my favorite number?")
print(f"  [green]agent:[/green] {say('What is my favorite number?', 'alice')}")

## Section 10: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements the same question on a different thr
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 4: The same question on a DIFFERENT thread
# Thread "bob" has its own history, which is empty — so the agent has no idea.
# This isolation is how one deployment serves many users safely.
print("\n[bold cyan]Thread 'bob' (a separate conversation):[/bold cyan]")
print("  [dim]user:[/dim]  What is my favorite number?")
print(f"  [green]agent:[/green] {say('What is my favorite number?', 'bob')}")

## Section 12: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 5: Inspect the stored history
# get_state() lets you read back whatever the checkpointer holds for a thread.
print("\n[bold cyan]Messages stored per thread:[/bold cyan]")
for thread_id in ("alice", "bob"):
    snapshot = agent.get_state({"configurable": {"thread_id": thread_id}})
    print(f"  [yellow]{thread_id}[/yellow]: {len(snapshot.values['messages'])} messages")

# %%